# WildSpeech M8.7 — Kaggle GPU + Google Drive edition

Original M8.7 scientific workflow preserved (A00–A16, frozen baselines, QA gates). Kaggle-specific local extraction and authenticated Drive checkpoints. Turn on Kaggle GPU and Internet and create the Kaggle secret RCLONE_CONFIG_B64. Never paste Drive tokens into code.

## 0. Authenticate Google Drive using Kaggle Secrets

On your trusted computer run rclone config, create a Drive remote called gdrive, and authorize it. Encode the resulting rclone.conf as base64 and save it as Kaggle Notebook secret RCLONE_CONFIG_B64. Grant this notebook secret access. The default Drive workspace path is gdrive:WildSpeech_M87_ACCESS; override with environment variable WILDSPEECH_REMOTE_WORKSPACE when necessary. This notebook downloads original private assets to Kaggle disk and syncs output batches back to the same authorized Drive workspace. No native Colab mount is used.
IMPORTANT: Existing M8.7 shards must restore BOTH .parquet and .npz because _load_existing verifies tensor pairs. This may require substantial Kaggle disk space; verify your session storage before running.


In [ ]:
import os,sys,json,subprocess,importlib,time,shutil,base64,zipfile,stat,urllib.request
from pathlib import Path
import pandas as pd,numpy as np
from IPython.display import display,Markdown,Image
from kaggle_secrets import UserSecretsClient
ON_COLAB=False
ROOT=Path("/kaggle/working")
assert ROOT.is_dir(), "This notebook requires Kaggle"
DRIVE=ROOT/"wildspeech_drive"
WORKSPACE=DRIVE/"WildSpeech_M87_ACCESS"
REMOTE_WORKSPACE=os.environ.get("WILDSPEECH_REMOTE_WORKSPACE","gdrive:WildSpeech_M87_ACCESS").rstrip("/")
RCFG=ROOT/".config/rclone/rclone.conf"
RCFG.parent.mkdir(parents=True,exist_ok=True)
def sh(args):
    subprocess.run([str(x) for x in args],check=True)
# Kaggle reports BackendError when a secret is missing or inaccessible.
# Never log the secret itself.
secret=os.environ.get("RCLONE_CONFIG_B64","").strip()
if not secret:
    try:
        secret=UserSecretsClient().get_secret("RCLONE_CONFIG_B64")
    except Exception as exc:
        if "No user secrets exist" in str(exc) or "RCLONE_CONFIG_B64" in str(exc):
            raise RuntimeError(
                "Missing/inaccessible Kaggle secret RCLONE_CONFIG_B64. "
                "Open Kaggle Notebook > Add-ons > Secrets, create that exact label "
                "with a base64-encoded authorized rclone.conf, enable notebook access, "
                "then rerun this cell. Do not paste credentials into code or output."
            ) from None
        raise
if not secret: raise RuntimeError("Kaggle secret RCLONE_CONFIG_B64 is blank")
try:
    decoded=base64.b64decode(secret,validate=True)
except Exception:
    raise RuntimeError("RCLONE_CONFIG_B64 must be valid base64 of rclone.conf") from None
if b"[gdrive]" not in decoded:
    raise RuntimeError("Decoded rclone.conf must contain a [gdrive] remote section")
RCFG.write_bytes(decoded)
del decoded,secret
RCFG.chmod(0o600)
os.environ["RCLONE_CONFIG"]=str(RCFG)
if not shutil.which("rclone"):
    archive=ROOT/"rclone.zip"
    urllib.request.urlretrieve("https://downloads.rclone.org/rclone-current-linux-amd64.zip",archive)
    with zipfile.ZipFile(archive) as z:
        member=next(x for x in z.namelist() if x.endswith("/rclone"))
        with z.open(member) as src,(ROOT/"rclone").open("wb") as dst: shutil.copyfileobj(src,dst)
    (ROOT/"rclone").chmod(0o700)
    os.environ["PATH"]=str(ROOT)+os.pathsep+os.environ.get("PATH","")
    archive.unlink()
def remote(subdir=""):
    return REMOTE_WORKSPACE+("/"+subdir.strip("/") if subdir else "")
sh(["rclone","lsf",REMOTE_WORKSPACE,"--max-depth","1"])
def pull_dir(subdir,patterns=None):
    dest=WORKSPACE/subdir
    dest.mkdir(parents=True,exist_ok=True)
    cmd=["rclone","copy",remote(subdir),str(dest),"--transfers","4","--checkers","8","--retries","4"]
    for p in patterns or []: cmd.extend(["--include",p])
    sh(cmd)
def push_dir(subdir):
    src=WORKSPACE/subdir
    if src.is_dir():
        sh(["rclone","copy",str(src),remote(subdir),"--transfers","2","--checkers","8","--retries","4"])
OFFICIAL_CSVS=("train_student_metadata.csv","test_student_metadata.csv","train_labels.csv","submission_format.csv","train_clip_metadata.csv","test_clip_metadata.csv")
pull_dir("WildSpeech",["*.csv"])
pull_dir("WildSpeech_M84/phase0")
pull_dir("WildSpeech_M86/m0pp/full")
for sub in ("manifests","features","qa","figures","evaluations"):
    try: pull_dir("WildSpeech_M87/speech/"+sub)
    except subprocess.CalledProcessError: print("Optional remote folder unavailable:",sub)
try: pull_dir("WildSpeech_M87/speech/shards")
except subprocess.CalledProcessError: print("No earlier phonetic shards")
assert all((WORKSPACE/"WildSpeech"/f).is_file() for f in OFFICIAL_CSVS)
print("Kaggle/Drive ready:",WORKSPACE,"remote:",REMOTE_WORKSPACE)


## 1. Install version-pinned M8.7 source

The phonetic model card verified transformers 4.57.6 and FP32 WavLM, and expects mono 16 kHz audio. Model weights will be fetched once from Hugging Face; **no competition audio is uploaded**. Use a separate runtime for the optional 1.7B word model.

In [ ]:
URL = "https://github.com/mybenkhadda/WildSpeech.git"
CHECKOUT = ROOT/"WildSpeech_git"
previous_sha = os.environ.get("WILDSPEECH_COMMIT", "")
if not (CHECKOUT / ".git").is_dir():
    if CHECKOUT.exists():
        raise RuntimeError(f"Existing non-Git directory: {CHECKOUT}")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    URL, str(CHECKOUT)], check=True)
else:
    subprocess.run(["git", "-C", str(CHECKOUT), "pull", "--ff-only",
                    "origin", "main"], check=True)
PKG = CHECKOUT / "WildSpeech_M8"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-e", str(PKG) + "[m87]"], check=True)
source = str(PKG / "src")
if source not in sys.path:
    sys.path.insert(0, source)
GIT_SHA = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"],
    text=True
).strip()
if (previous_sha and previous_sha != GIT_SHA and
        any(x.startswith("wildspeech_m87.") for x in sys.modules)):
    raise RuntimeError(
        f"Source updated from {previous_sha[:8]} to {GIT_SHA[:8]} with old "
        "modules still loaded. Use Runtime > Restart session, then run from cell 0."
    )
os.environ["WILDSPEECH_COMMIT"] = GIT_SHA

import torch
# Silero VAD/NumPy signal processing run on CPU; avoid oversubscribing
# a 2-core Colab host while the GPU handles WavLM in FP32.
torch.set_num_threads(min(2, max(1, os.cpu_count() or 1)))
print("Source commit:", GIT_SHA)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU disabled: extraction requires a CUDA runtime")

## 2. Paths, private local staging, and safe run controls

Auto-detects the original Drive or `WildSpeech_M87_ACCESS`. The original output location is **never reset**. For speed, Colab copies the 1.91 GB audio TARs to ephemeral local disk once and writes/compresses each shard locally before uploading it to Drive. Each shard is uploaded completely before moving to the next one; a disconnect preserves completed shards. Neither the model precision nor the official fold split changes.

Review model licenses and competition rules before setting `LICENSE_ACKNOWLEDGED=True`. Start with phonetic extraction only; Word ASR is a separate runtime, and evaluation requires full extraction.

In [ ]:
from wildspeech_m87.contracts import Config, ASSETS, EXPERIMENTS, FULL_FOLD_HASH
from wildspeech_m84.data import locate_data_root, load_official

# Explicit override: do NOT autodiscover an unrelated MyDrive/WildSpeech.
DATA_ROOT = locate_data_root(DRIVE, override=str(WORKSPACE / "WildSpeech"))
PHASE0_ROOT = WORKSPACE / "WildSpeech_M84" / "phase0"
M86_ROOT = WORKSPACE / "WildSpeech_M86" / "m0pp" / "full"
OUTPUT = WORKSPACE / "WildSpeech_M87" / "speech"

TRAIN_TAR = str(DATA_ROOT / "train_audio.tar")
TEST_TAR = str(DATA_ROOT / "test_audio.tar")
TRAIN_AUDIO_DIR = ""
TEST_AUDIO_DIR = ""

# Must be manually set after license and competition-rule review.
LICENSE_ACKNOWLEDGED=False
RUN_GPU_EXTRACT=False              # set True to resume all missing clips
RUN_WORD_ASR=False                # use separate compatible runtime
RUN_EVALUATION=False              # only after full cache coverage
MODE="SCREEN"                     # set FULL for 100 official outer folds
MAX_CLIPS=96                      # 2 shards between Drive checkpoints
VAD_METHOD="silero"               # keep unchanged to reuse the cache
BATCH_SIZE=2                      # safe T4 baseline; compare 4 only after profiling
SHARD_SIZE=48
PROFILE_EVERY=4

STAGE_AUDIO_LOCALLY=False
STAGE_SHARDS_LOCALLY=False
LOCAL_AUDIO_ROOT=ROOT/"wildspeech_audio"
LOCAL_SHARD_ROOT=ROOT/"wildspeech_shard_stage"

print("Data root:", DATA_ROOT)
print("M8.4 reference:", PHASE0_ROOT)
print("M8.6 FULL:", M86_ROOT)
print("Preserved output:", OUTPUT)
print("Train TAR:", TRAIN_TAR)
print("Test TAR:", TEST_TAR)
display(pd.DataFrame([{"asset": name, **details} for name, details in ASSETS.items()]))

## 3. Verify data and fixed outer folds

Read original 541 training students, 361 test students, 14,704 training clips and 9,814 test clips. Read exact Phase 0 folds and M8.6 I5/I4 OOF, not new splits.

In [ ]:
from wildspeech_m87.contracts import validate_baselines,clip_inventory
DATA=load_official(DATA_ROOT)
FOLDS,REFERENCE,INVENTORY,ANCHORS=validate_baselines(
    DATA,PHASE0_ROOT,M86_ROOT,official=True)
display(pd.DataFrame([ANCHORS]).T)
assert ANCHORS["fold_hash"]==FULL_FOLD_HASH
print("Folds:",FOLDS.groupby(["scheme","repeat"]).size().to_dict())
print("Train/test audio inventory:",INVENTORY.groupby(["split","task"]).size().to_dict())

## 4. Locate audio archives (or set explicit directory paths)

The folder audit below searches only bounded likely locations and does not scan the entire Drive recursively. Set paths manually if ambiguous.

In [ ]:
for folder in dict.fromkeys([
    DATA_ROOT, DATA_ROOT.parent, WORKSPACE, OUTPUT,
    DRIVE / "WildSpeech", LOCAL_AUDIO_ROOT
]):
    if folder.is_dir():
        items = [x.name for x in list(folder.iterdir())[:100]
                 if x.suffix.lower() in (".tar", ".flac", ".wav", ".parquet")
                 or "audio" in x.name.lower()]
        print(folder, "->", items[:30])
print("Existing completed shards:",
      len(list((OUTPUT / "shards").glob("*_phonetic.parquet")))
      if (OUTPUT / "shards").is_dir() else 0)

## 5. Kaggle local archives and pinned fingerprint

The two audio TARs are pulled from your private Drive to Kaggle local disk, not passed to any external inference service. Shard creation occurs on local storage. Frozen reference files and extraction fingerprint are retained.

In [ ]:
from wildspeech_m84.data import digest_bytes
for _tar in ("train_audio.tar","test_audio.tar"):
    _dest=DATA_ROOT/_tar
    if not _dest.is_file():
        _dest.parent.mkdir(parents=True,exist_ok=True)
        sh(["rclone","copyto",remote("WildSpeech/"+_tar),str(_dest),"--retries","5"])


# Explicit checks before an expensive GPU run.
required = {
    "M8.4 frozen fold manifest": PHASE0_ROOT / "manifests" / "fold_manifest.json",
    "M8.4 source manifest": PHASE0_ROOT / "manifests" / "data_manifest.json",
    "M8.4 folds": PHASE0_ROOT / "folds" / "folds.parquet",
    "M8.4 parity": PHASE0_ROOT / "parity" / "M84_RESULTS.json",
    "M8.6 FULL report": M86_ROOT / "reports" / "M86_RESULTS.json",
    "train_audio.tar": Path(TRAIN_TAR),
    "test_audio.tar": Path(TEST_TAR),
}
missing = [f"{k}: {v}" for k, v in required.items() if not v.is_file()]
if missing:
    raise FileNotFoundError("Missing private inputs:\n" + "\n".join(missing))

OUTPUT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT / "manifests" / "extraction.json"
existing_manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
PINNED_REVISION = existing_manifest.get("resolved_phonetic_revision", "")
print("Existing model revision:", PINNED_REVISION or "will resolve on first run")

USE_LOCAL_AUDIO = False
if ON_COLAB and STAGE_AUDIO_LOCALLY and RUN_GPU_EXTRACT:
    source_tars = [Path(TRAIN_TAR), Path(TEST_TAR)]
    needed = sum(p.stat().st_size for p in source_tars)
    free = shutil.disk_usage("/content").free
    # Do not fill the Colab instance; keep at least 2 GiB free.
    if free > needed + 2 * 1024**3:
        LOCAL_AUDIO_ROOT.mkdir(parents=True, exist_ok=True)
        staged = []
        for src in source_tars:
            dest = LOCAL_AUDIO_ROOT / src.name
            if not dest.is_file() or dest.stat().st_size != src.stat().st_size:
                partial = dest.with_name(dest.name + ".partial")
                with src.open("rb") as inp, partial.open("wb") as out:
                    shutil.copyfileobj(inp, out, length=16 * 1024**2)
                if partial.stat().st_size != src.stat().st_size:
                    raise IOError(f"Truncated local TAR copy: {src}")
                os.replace(partial, dest)
            staged.append(dest)
        TRAIN_TAR, TEST_TAR = map(str, staged)
        USE_LOCAL_AUDIO = True
    else:
        print("Insufficient local disk space; reading TAR directly from Drive.")

CONFIG = Config(
    drive_root=str(DRIVE), data_root=str(DATA_ROOT),
    output_root=str(OUTPUT), train_tar=TRAIN_TAR, test_tar=TEST_TAR,
    train_audio_dir=TRAIN_AUDIO_DIR, test_audio_dir=TEST_AUDIO_DIR,
    phonetic_revision=PINNED_REVISION,
    acknowledge_license_review=LICENSE_ACKNOWLEDGED,
    batch_size=BATCH_SIZE, mode=MODE
)
if RUN_GPU_EXTRACT:
    CONFIG.validate()
    assert torch.cuda.is_available(), "Enable a GPU runtime before extraction."
print("Audio source:", "local ephemeral TARs" if USE_LOCAL_AUDIO else "private Drive TARs")
print("Cache output:", CONFIG.output())
print("N experiment arms:", len(EXPERIMENTS), "| Source SHA:", GIT_SHA)
print("Read-only frozen source files stay untouched.")

## 6. TRAIN-first GPU extraction, resumable and checkpointed

Original train/test inventory and source model revision remain unchanged. Prior completed shard IDs are skipped. After every MAX_CLIPS batch, all completed shard pairs and manifests are backed up to the authorized Google Drive remote. An interrupted Kaggle session may lose the current unfinished batch; earlier synced batches persist.

In [ ]:
import wildspeech_m87.gpu_cache as gpu_cache
from wildspeech_m87.gpu_cache import _load_existing
from wildspeech_m84.data import atomic_json
from concurrent.futures import ThreadPoolExecutor
import pyarrow.parquet as pq

# Keep the original model, clip schemas and immutable cache fingerprint;
# modify only which incomplete split is processed first.
source_file=Path(gpu_cache.__file__)
source_code=source_file.read_text()
old='todo=inventory[~inventory.uid.isin(seen)].sort_values(\n        ["split","duration_sec","uid"],kind="stable")'
new='todo=inventory[~inventory.uid.isin(seen)].copy()\n    todo["_priority"]=todo["split"].map({"train":0,"test":1}).fillna(2)\n    todo=todo.sort_values(["_priority","duration_sec","uid"],kind="stable").drop(columns="_priority")'
if old in source_code:
    source_file.write_text(source_code.replace(old,new,1))
    importlib.invalidate_caches()
    gpu_cache=importlib.reload(gpu_cache)
elif new not in source_code:
    raise RuntimeError("Extraction source changed; inspect split-order patch")

# Fast metadata-only summary for checkpoint batches (no full-feature rewrite).
def fast_summary(root,inventory,strict=True):
    paths=sorted((Path(root)/"shards").glob("*_phonetic.parquet"))
    if not paths: raise RuntimeError("No phonetic shards")
    with ThreadPoolExecutor(max_workers=4) as pool:
        parts=list(pool.map(lambda p:pq.read_table(p,columns=["uid","status"]).to_pandas(),paths))
    rows=pd.concat(parts,ignore_index=True)
    if rows.uid.duplicated().any(): raise RuntimeError("Duplicate cached UID")
    if set(rows.uid)-set(inventory.uid): raise RuntimeError("Foreign UID in cache")
    counts=dict(expected=len(inventory),accounted=len(rows),
                valid=int(rows.status.eq("VALID").sum()),
                failed=int(rows.status.eq("FAILED").sum()),
                missing=len(inventory)-len(rows))
    if strict and counts["missing"]: raise RuntimeError(f"Incomplete inventory: {counts}")
    atomic_json(Path(root)/"manifests"/"coverage.json",counts)
    return counts
gpu_cache.summarize_cache=fast_summary

def sync_checkpoint():
    for part in ("shards","manifests","features"):
        push_dir("WildSpeech_M87/speech/"+part)
    print("Drive checkpoint uploaded.")

if RUN_GPU_EXTRACT:
    assert LICENSE_ACKNOWLEDGED, "Review model license/competition privacy rules first"
    assert torch.cuda.is_available(), "Select Kaggle GPU accelerator"
    assert MAX_CLIPS>0, "Kaggle MAX_CLIPS must be >0"
    try:
        while True:
            seen=_load_existing(OUTPUT)
            missing=len(INVENTORY)-len(seen)
            print(f"Already cached={len(seen):,} missing={missing:,}")
            if not missing: break
            coverage=gpu_cache.extract(
                CONFIG,INVENTORY,max_clips=min(MAX_CLIPS,missing),
                shard_size=SHARD_SIZE,vad_method=VAD_METHOD,
                staging_dir=None,profile_every=PROFILE_EVERY)
            print("Coverage:",coverage)
            sync_checkpoint()
    finally:
        sync_checkpoint()
else:
    print("Inference paused. Review license then set LICENSE_ACKNOWLEDGED=True and RUN_GPU_EXTRACT=True")


## 7. Validate local cache and materialize complete features once

In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
CACHE_FEATURES=None
shards=sorted((OUTPUT/"shards").glob("*_phonetic.parquet"))
if shards:
    with ThreadPoolExecutor(max_workers=4) as pool:
        frames=list(tqdm(pool.map(pd.read_parquet,shards),total=len(shards),desc="Local cache shards"))
    CACHE_FEATURES=pd.concat(frames,ignore_index=True)
    del frames
    if CACHE_FEATURES.uid.duplicated().any(): raise RuntimeError("Duplicate UIDs")
    if set(CACHE_FEATURES.uid)-set(INVENTORY.uid): raise RuntimeError("Foreign cache UIDs")
    stats={"expected":len(INVENTORY),"accounted":len(CACHE_FEATURES),
           "valid":int(CACHE_FEATURES.status.eq("VALID").sum()),
           "failed":int(CACHE_FEATURES.status.eq("FAILED").sum()),
           "missing":len(INVENTORY)-len(CACHE_FEATURES)}
    print("Coverage:",stats)
    if stats["missing"]==0:
        target=OUTPUT/"features"/"phonetic_clip_features_PRIVATE.parquet"
        target.parent.mkdir(parents=True,exist_ok=True)
        tmp=target.with_name(target.name+".partial")
        CACHE_FEATURES.to_parquet(tmp,index=False)
        os.replace(tmp,target)
        push_dir("WildSpeech_M87/speech/features")
    else:
        print("PARTIAL coverage: full A00–A16 evaluation blocked")
    if (OUTPUT/"features"/"word_asr_PRIVATE.parquet").exists():
        word=pd.read_parquet(OUTPUT/"features"/"word_asr_PRIVATE.parquet")
        if word.uid.duplicated().any(): raise RuntimeError("Duplicate word UIDs")
        if "word_partial_available" in CACHE_FEATURES and "word_partial_available" in word:
            CACHE_FEATURES=CACHE_FEATURES.drop(columns=["word_partial_available"])
        CACHE_FEATURES=CACHE_FEATURES.merge(word,on="uid",how="left",validate="one_to_one")
    display(CACHE_FEATURES.groupby(["split","status"]).size().to_frame("n"))
else: print("No phonetic cache yet")


## 8. C3–C6 evidence figures: coverage, timing, uncertainty, correctness strata

Generate figures ONLY from the actual cached feature table. All artifacts stay on Drive.

In [ ]:
from wildspeech_m87.diagnostics import extraction_atlas
if CACHE_FEATURES is not None:
    ATLAS=extraction_atlas(OUTPUT,INVENTORY,CACHE_FEATURES)
    print("Evidence atlas:",ATLAS)
    for fn in ("01_cache_coverage.png","02_duration_by_task.png",
               "03_localization_confidence.png","04_whole_vs_segment_phone_edits.png",
               "05_ctc_segment_delta.png","06_grading_within_correctness.png"):
        path=OUTPUT/"figures"/fn
        if path.exists():display(Image(filename=str(path)))
else:print("Atlas skipped until extraction cache exists.")

## 9. Phonetic target mapping and explicit failures

Inspect unknown G2P mappings and IPA tokenizer coverage. No forced alignment should silently invent unsupported phonemes.

In [ ]:
targets=OUTPUT/"manifests"/"target_registry_PRIVATE.parquet"
if targets.exists():
    t=pd.read_parquet(targets)
    display(t.groupby(["task","reason"]).size().to_frame("n"))
    print("Target-valid share:",float(t.valid.mean()))
    display(t[~t.valid][["task","expected_text","reason"]].head(30))
else:print("Run phonetic extraction to generate target registry.")

## 10. Training-only waveform localization QA

The plots and manual QA must use training clips ONLY. Test recordings must not be listened to or manually transcribed.

In [ ]:
from wildspeech_m87.localization_qa import create_template,evaluate_qa
QA_FILE=OUTPUT/"qa"/"training_qa_template_PRIVATE.csv"
if QA_FILE.exists():
    print("Existing reviewed/unreviewed QA file preserved:",QA_FILE)
elif CACHE_FEATURES is None: print("QA deferred: no cache")
else:
    train_ids=set(INVENTORY.loc[INVENTORY.split=="train","uid"])
    valid=CACHE_FEATURES.loc[
        CACHE_FEATURES.uid.isin(train_ids) &
        CACHE_FEATURES.status.astype(str).str.strip().str.upper().eq("VALID")]
    print("VALID training clips:",len(valid))
    if valid.empty: print("QA deferred until training extraction")
    else:
        norm=CACHE_FEATURES.copy()
        norm["status"]=norm.status.astype(str).str.strip().str.upper()
        qa=create_template(DATA,INVENTORY,norm,QA_FILE,n=150)
        print("Training-only QA:",QA_FILE,len(qa))
        push_dir("WildSpeech_M87/speech/qa")
if QA_FILE.exists():
    result=evaluate_qa(QA_FILE,OUTPUT/"qa")
    print("QA status:",result)
    if result.get("status")=="NOT_REVIEWED":
        print("Localization threshold NOT established without human review")
    push_dir("WildSpeech_M87/speech/qa")


## 11. Optional training-only waveform figures

Create the 12 example waveform plots using the training archive only; they show the predicted response region and support manual review.

In [ ]:
MAKE_TRAINING_QA_WAVEFORMS=False
if MAKE_TRAINING_QA_WAVEFORMS and CACHE_FEATURES is not None:
    from wildspeech_m87.gpu_cache import create_sources
    from wildspeech_m87.diagnostics import qa_training_waveforms
    s=create_sources(CONFIG,OUTPUT)["train"]
    n=qa_training_waveforms(OUTPUT,INVENTORY,CACHE_FEATURES,s,n=12)
    print("Training QA figures:",n)

## 12. Optional local Pasketti Word inference for sentence repetition

This model uses Qwen3-ASR-1.7B with qwen_asr[vllm] 0.0.6, typically requiring a separate compatible Linux/CUDA runtime. Do not run both heavy Pasketti models simultaneously. If absent, A07 is reported NOT_ELIGIBLE; it is never assigned a fake score.

In [ ]:
if RUN_WORD_ASR:
    assert LICENSE_ACKNOWLEDGED
    from wildspeech_m87.gpu_cache import extract_words, summarize_cache
    full = summarize_cache(OUTPUT, INVENTORY, strict=True)
    print("Phonetic cache complete:", full)
    from wildspeech_m87.gpu_cache import extract_words
    result = extract_words(
        CONFIG, INVENTORY, max_clips=MAX_CLIPS, sentence_only=True
    )
    print("Word ASR cached:", result)
else:
    print("Pasketti Word disabled; A07 requires a separate supported runtime.")

## 13. Task-specific phonetic and word partial-credit descriptions

Inspect real correlations with observed item_score, emphasizing *graded scores within each correctness stratum*, rather than reporting ASR accuracy as future-reading prediction.

In [ ]:
if (OUTPUT/"features"/"phonetic_clip_features_PRIVATE.parquet").exists():
    c=load_features(OUTPUT)
    fields=[x for x in ("child_phone_edit_rate","whole_phone_edit_rate",
          "task_phone_graded_accuracy","child_target_logp_per_phone",
          "word_coverage","word_lcs_ratio","clip_pred_correct") if x in c]
    if fields:
        display(c.groupby("task")[fields].agg(["mean","std","count"]).round(4))
else:print("Task plots require real cached phonetic results.")

## 14. Predeclared A00–A16 experiment registry

Includes whole vs child target scoring, entropy, nuisance-only controls, random-segment null, within-correctness, task-specific partial credit, grouped clip correctness, unscored recovery, confidence weighting, layer sketches, residual arm, and proper inner-OOF stacking.

In [ ]:
from wildspeech_m87.evaluation import experiment_plan
PLAN=pd.DataFrame(experiment_plan())
display(PLAN[["comparison_id","candidate","model","negative_control",
              "main_hypothesis"]])
assert len(PLAN)==17
print("Predeclared primary hypotheses:",PLAN.loc[PLAN.main_hypothesis,"candidate"].tolist())

## 15. Run A00–A16 matched-fold evaluation

SCREEN: first 5 student + 5 school folds; full model outcomes are not confirmatory. FULL: 50 student and 50 school-disjoint folds with 3 nested student folds. All model training uses only outer-training labels. Completed fold checkpoints resume automatically.

In [ ]:
if RUN_EVALUATION:
    assert MODE in ("SCREEN", "FULL")
    from wildspeech_m87.gpu_cache import summarize_cache
    complete = summarize_cache(OUTPUT, INVENTORY, strict=True)
    assert complete["missing"] == 0, "Complete phonetic extraction first."
    from wildspeech_m87.evaluation import evaluate
    RESULTS = evaluate(
        drive_root=DRIVE, data_root=str(DATA_ROOT),
        cache_root=str(OUTPUT), phase0_root=str(PHASE0_ROOT),
        m86_root=str(M86_ROOT), mode=MODE
    )
    print("Decision:", RESULTS["decision"], "winner:", RESULTS["winner"])
    print("Skipped arms:", RESULTS["skipped"])
else:
    print("Full evaluation disabled until complete cache + localization QA.")

## 16. Models ranked by student and school R²

Keep original M8.6 I5 and I4 metrics in the same table. A4/A5/A15 are NEGATIVE CONTROLS, not promotion candidates.

In [ ]:
EVAL_DIR=OUTPUT/"evaluations"/MODE.lower()
RESULT_JSON=EVAL_DIR/"reports"/"M87_RESULTS.json"
if RESULT_JSON.exists():
    RESULTS=json.loads(RESULT_JSON.read_text())
    table=pd.DataFrame(RESULTS["model_metrics"])
    show=[c for c in ("model","student_pooled_r2","school_pooled_r2",
                       "student_mae","student_rmse","mean_fold_delta_r2",
                       "win_fraction","corrected_t","engineering_eligible",
                       "science_gate","status","reason") if c in table]
    display(table[show].round(5))
else:print("No outcome R² results yet.")

## 17. Frozen-fold diagnostic plots and transfer frontier

In [ ]:
for filename in ("07_model_student_school_r2.png",
                 "08_paired_delta_r2.png","09_transfer_frontier.png",
                 "10_baseline_prediction_scatter.png"):
    path=EVAL_DIR/"figures"/filename
    if path.exists():
        print(filename)
        display(Image(filename=str(path)))

## 18. Paired R² against I5, instrument vetoes and robustness

Report win rate, corrected Nadeau–Bengio statistic, school pooled ΔR² and per-instrument R² changes. The corrected statistic is approximate, particularly after exploring many model families.

In [ ]:
paired=EVAL_DIR/"tables"/"07_paired_science_gate.csv"
if paired.exists():
    p=pd.read_csv(paired)
    display(p.round(6))
    print("No post-hoc changes to preregistered statistical thresholds.")
else:print("Paired comparison table requires FULL execution.")

## 19. Training-only outcome residual and school transfer atlas

Observe whether child speech features predict future reading residuals left by I5. The student embedding used for predictive claims must be generated out-of-fold.

In [ ]:
for filename in ("05_student_oof_residuals_PRIVATE.csv",
                 "06_within_correctness_PRIVATE.csv"):
    p=EVAL_DIR/"tables"/filename
    if p.exists():
        print(filename)
        display(pd.read_csv(p).head(15))

## 20. Perturbation-stability study (TRAIN clips only)

Optional: reread up to 45 TRAIN recordings and recompute phonetic evidence under 0.9× and 1.1× speed changes and added noise. These are stress tests; do not claim invariance is guaranteed.

In [ ]:
RUN_STABILITY=False
if RUN_STABILITY:
    assert LICENSE_ACKNOWLEDGED
    from wildspeech_m87.phonetic_asr import PhoneticASR
    from wildspeech_m87.gpu_cache import create_sources
    from wildspeech_m87.targets import TargetRegistry, normalize_text
    from wildspeech_m87.stability import run_stability
    model = PhoneticASR(
        revision=CONFIG.phonetic_revision, layers=(6,), max_seconds=30.
    )
    source = create_sources(CONFIG, OUTPUT)["train"]
    registry = TargetRegistry(model.processor.tokenizer).make(
        INVENTORY[INVENTORY.split == "train"]
    )
    mapping = {
        (r.task, normalize_text(r.expected_text)): r.ids
        for r in registry.itertuples(index=False) if r.valid
    }
    STABILITY = run_stability(INVENTORY, source, model, mapping,
                             OUTPUT / "stability", max_clips=45)
    print(STABILITY)

## 21. Statistical decision and detailed science report

If the child localization QA has not been reviewed and passed, the output is *experimental evidence*, not a verified child-specific measurement.

In [ ]:
REPORT=EVAL_DIR/"reports"/"M87_DECISION.md"
if REPORT.exists():display(Markdown(REPORT.read_text(encoding="utf-8")))
else:print("No FULL science report until A00–A16 experiments have executed.")

## 22. Conditional final submission exporter — no automatic online upload

Requires FULL engineering gate and reviewed localization accuracy/false adult-selection checks. Independently verifies test singletons and row permutations; produces a private CSV/checksum only.

In [ ]:
RUN_SUBMISSION_EXPORT=False
if RUN_SUBMISSION_EXPORT:
    from wildspeech_m87.submission import export_if_eligible
    manifest=export_if_eligible(
        drive_root=DRIVE,cache_root=OUTPUT,
        eval_root=OUTPUT/"evaluations"/"full",
        data_root=str(DATA_ROOT),rasch_iters=7,require_qa=True)
    print("Verified candidate:",manifest["file"],"SHA:",manifest["sha256"])
else:print("Submission not exported or submitted online.")

## 23. Private artifact inventory

In [ ]:
for folder in ("manifests","indexes","shards","features","figures","qa",
               "evaluations/screen","evaluations/full","stability"):
    path=OUTPUT/folder
    if path.is_dir():
        files=list(path.iterdir())
        print(folder,":",len(files),"entries",
              [f.name for f in files[:5]])
    else:print(folder,": not generated")

## 24. Handoff to M8.8

If A02 (child-only evidence) improves over A01 (whole-clip evidence), and the QA gate is satisfied, prioritize sentence-repetition partial credit and task-specific phonetic error topology.

If no speech family improves I5/I4, preserve the screening baselines and investigate localization, ASR reliability, item alignment and recording nuisances first. Do not train another large end-to-end student Transformer based on a null result.

No results in this notebook are fabricated: all tables and plots are conditional on actual private data extraction and completed experiments.